# demo_prestructured

> Pre-structured (setpoint) policy with bump jumps, reproducing the shape of Fig 2h (Dan et al., Neuron 2024).

This notebook follows `specs/prestructured/`. **How to run:** [`specs/prestructured/RUN.md`](../specs/prestructured/RUN.md). Normal tests (`nbdev-test`) run 2 runs of 240 s per condition; the full version (`nbdev-test --flags slow`, or running the notebook by hand) uses 20 runs and checks acceptance criteria AC-8 to AC-10.

Figures and metrics are written under `results/demo_prestructured/` (gitignored; layout in RUN.md).

## Setup

Table 1 parameters; the reward goal is 180 degrees.

In [ ]:
from math import pi
import numpy as np
import matplotlib.pyplot as plt
from explRL.params import Params
from explRL.prestructured import angdiff, turnProb, driftRate, jumpProb, effectivePolicy
from explRL.prestructured_train import runManyPrestructured, relHeading, heldGoal
from explRL.train import save_demo_figure, save_demo_metrics

DN = 'demo_prestructured'
params, noJumps = Params(), Params(GJ=0.0, BJ=0.0)
seed, goal, nRuns = 0, pi, 2
artifact_mode = 'quick'
deg = np.linspace(-180, 180, 361)


## Policy curves (Eqs. 20-21)

As a function of compass heading minus goal heading.

In [ ]:
th = goal + np.deg2rad(deg)
fig, axes = plt.subplots(1, 3, figsize=(12, 2.5))
for ax, f, title in zip(axes, [turnProb, driftRate, jumpProb], ['CW saccade probability dS', 'drift rate nuF', 'jump probability pJ']):
    ax.plot(deg, f(th, goal, params)); ax.set_xlabel('compass - goal (deg)'); ax.set_title(title)
plt.tight_layout()
save_demo_figure('policy_curves', fig=fig, mode=None, demo_name=DN)
plt.show()


## Analytic Fig 2h (Eq. 23)

What an observer sees in arena coordinates, with and without bump jumps.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
for p, label in [(params, 'jumps on'), (noJumps, 'jumps off')]:
    ev = effectivePolicy(th, goal, p)
    axes[0].plot(deg, ev['fixDur'], label=label); axes[1].plot(deg, ev['turnBias'], label=label)
for ax, title in zip(axes, ['mean fixation duration (s)', 'turn bias (+ = CW)']):
    ax.set_xlabel('arena - goal (deg)'); ax.set_title(title); ax.legend()
plt.tight_layout()
save_demo_figure('fig2h_analytic', fig=fig, mode=None, demo_name=DN)
plt.show()


## Simulated runs

Three conditions: fixed goal with jumps (Fig 2h), goal learning without jumps, and goal learning with jumps. The quick cell keeps normal tests fast; the slow cell reruns with 20 runs.

In [ ]:
runs_fix = runManyPrestructured(params, goal, nRuns, seed=seed, learn=False, thetaG0=goal)
runs_learn = runManyPrestructured(noJumps, goal, nRuns, seed=seed, learn=True)
runs_jump = runManyPrestructured(params, goal, nRuns, seed=seed, learn=True)


In [ ]:
#| slow
artifact_mode, nRuns = 'full', 20
runs_fix = runManyPrestructured(params, goal, nRuns, seed=seed, learn=False, thetaG0=goal)
runs_learn = runManyPrestructured(noJumps, goal, nRuns, seed=seed, learn=True)
runs_jump = runManyPrestructured(params, goal, nRuns, seed=seed, learn=True)


In [ ]:
def pooled(runs):
    "Fixation durations and relative headings (degrees) of all events of all runs."
    fix = np.concatenate([r['history'].fixDur for r in runs])
    rel = np.concatenate([relHeading(r['history'].thetaA, goal) for r in runs])
    return fix, rel

def distGoal(g): return np.abs(angdiff(g, goal))
def distNearer(g): return np.minimum(distGoal(g), np.abs(angdiff(g, goal + pi)))

grid = np.arange(0, params.Ttot, 1.0)
lastQ = grid >= 0.75 * params.Ttot

def distTrace(r, dist):
    "Distance of the goal heading in effect at each grid time (1 s steps)."
    return dist(heldGoal(r['history'], r['thetaG0'], grid, params.tS))

def startAndLastQuarter(runs, dist):
    "Distance at the start and time-averaged over the last quarter of the run, averaged over runs."
    start = np.mean([dist(r['thetaG0']) for r in runs])
    last = np.mean([distTrace(r, dist)[lastQ].mean() for r in runs])
    return float(start), float(last)

fix, rel = pooled(runs_fix)
near = float(fix[np.abs(rel) <= 30].mean())
anti = float(fix[np.abs(rel) >= 150].mean())
side = float(fix[(np.abs(rel) >= 60) & (np.abs(rel) <= 120)].mean())
ac9 = startAndLastQuarter(runs_learn, distGoal)
ac10 = startAndLastQuarter(runs_jump, distNearer)
print(f'fixation goal {near:.2f} s, anti-goal {anti:.2f} s, side {side:.2f} s')
print(f'no jumps |thetaG - goal|: {ac9[0]:.2f} -> {ac9[1]:.2f} rad; jumps, nearer goal: {ac10[0]:.2f} -> {ac10[1]:.2f} rad')


Simulated readout of the fixed-goal runs, with the Eq. 23 curve for comparison. The anti-goal peak is higher than Eq. 23 predicts because the compass offset persists between jumps (spec design D4).

In [ ]:
ro = [r['readout'] for r in runs_fix]
centers = ro[0]['centers']
fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
ev = effectivePolicy(th, goal, params)
for ax, key, title in zip(axes, ['fixDur', 'turnBias'], ['mean fixation duration (s)', 'turn bias (+ = CW)']):
    ax.plot(centers, np.nanmean([r[key] for r in ro], 0), 'o-', label='simulated')
    ax.plot(deg, ev[key], 'k--', alpha=0.5, label='Eq. 23')
    ax.set_xlabel('arena - goal (deg)'); ax.set_title(title); ax.legend()
plt.tight_layout()
save_demo_figure('fig2h_simulated', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()


## Goal-heading learning

In [ ]:
d = np.mean([distTrace(r, distGoal) for r in runs_learn], 0)
fig = plt.figure(figsize=(5, 2.5))
plt.plot(grid, d); plt.xlabel('time (s)'); plt.ylabel('|thetaG - goal| (rad)'); plt.title('learning, no jumps')
plt.tight_layout()
save_demo_figure('learning_nojump', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()


In [ ]:
d = np.mean([distTrace(r, distNearer) for r in runs_jump], 0)
fig = plt.figure(figsize=(5, 2.5))
plt.plot(grid, d); plt.xlabel('time (s)'); plt.ylabel('distance to nearer goal (rad)'); plt.title('learning, with jumps')
plt.tight_layout()
save_demo_figure('learning_jump', fig=fig, mode=artifact_mode, demo_name=DN)
plt.show()


In [ ]:
metrics = dict(mode=artifact_mode, seed=seed, goal_rad=goal, nRuns=nRuns, Ttot_s=params.Ttot)
if artifact_mode == 'full':
    metrics.update(AC8_fix_goal_s=near, AC8_fix_antigoal_s=anti, AC8_fix_side_s=side,
                   AC9_dist_start_rad=ac9[0], AC9_dist_last_quarter_rad=ac9[1],
                   AC10_nearer_start_rad=ac10[0], AC10_nearer_last_quarter_rad=ac10[1],
                   jump_rate_fixed_goal=float(np.mean([r['jumpRate'] for r in runs_fix])))
save_demo_metrics(metrics, mode=artifact_mode, demo_name=DN)


## Acceptance checks (AC-8 to AC-10)

In [ ]:
#| slow
assert near > side and anti > side                                              # AC-8
assert ac9[1] < ac9[0]                                                          # AC-9
assert all(np.isfinite(r['history'].thetaG).all() for r in runs_learn + runs_jump)  # AC-10 (finiteness only)


## Summary

With the paper's Table 1 setpoint parameters, the decided signs (spec Q1, Q2) and no tuning, 20 runs of 240 s (seeds 0-19) gave:

- **AC-8**: mean fixation 11.5 s within ±30 degrees of the goal and 10.6 s within ±30 degrees of the anti-goal, versus 2.5 s at 60-120 degrees from the goal: the readout is bimodal, as in Fig 2h.
- **AC-9**: without jumps, the distance from the goal heading to the reward goal fell from 1.67 rad at the start to 0.51 rad (time average over the last quarter).
- **AC-10** (reported only): with jumps, the distance to the nearer of goal / anti-goal went from 0.84 to 0.74 rad, so no clear convergence.

The simulated anti-goal peak is much higher than Eq. 23 predicts (3.2 s), because the compass offset persists between jumps. All three conditions run in about a second.